# Solution — Pratique ML — Classification (scoring de risque de défaut)


**Objectif entretien Data Scientist / ML Engineer.**

Cible : `defaulted` (1 = l'entreprise fait défaut dans les 12 mois, 0 sinon).

## Mode d'emploi
- Remplis chaque cellule marquée `# TODO`. L'ordre suit un vrai pipeline d'entretien.
- Le **résultat attendu** est indiqué en commentaire au-dessus de chaque TODO.
- Corrigé complet : [`solutions/classification_xgboost.py`](solutions/classification_xgboost.py). Ne le regarde **qu'après** avoir tenté.
- Chronomètre-toi : ~40 min cible.

## Plan
0. Setup (fourni) · 1. Exploration · 2. Split stratifié · 3. Préprocessing · 4. Baseline LogisticRegression · 5. RandomForest + XGBoost · 6. Déséquilibre · 7. Métriques · 8. Cross-validation · 9. Seuil métier

## 0. Setup (fourni — exécute simplement)
Charge les données ; régénère le CSV automatiquement s'il manque.

In [ ]:
from pathlib import Path
import subprocess, sys
import numpy as np
import pandas as pd

RANDOM_STATE = 42
TARGET = "defaulted"
CATEGORICAL_FEATURES = ["country", "sector"]
NUMERIC_FEATURES = ["revenue", "debt_ratio", "days_late", "num_employees", "credit_score", "years_in_business"]
FEATURES = CATEGORICAL_FEATURES + NUMERIC_FEATURES

DATA_PATH = Path("data/credit_risk.csv")
if not DATA_PATH.exists():
    if Path("generate_dataset.py").exists():
        subprocess.run([sys.executable, "generate_dataset.py"], check=True)
    else:
        # Fallback pour rendre le notebook exécutable même si le générateur n'est pas fourni.
        DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
        rng = np.random.default_rng(RANDOM_STATE)
        n = 5000
        tmp = pd.DataFrame({
            "country": rng.choice(["FR", "MA", "ES", "DE", "US"], size=n),
            "sector": rng.choice(["retail", "industry", "services", "tech", "finance"], size=n),
            "revenue": rng.lognormal(mean=12, sigma=0.9, size=n),
            "debt_ratio": rng.beta(2, 5, size=n),
            "days_late": rng.poisson(8, size=n),
            "num_employees": rng.integers(5, 2000, size=n),
            "credit_score": rng.normal(650, 80, size=n).clip(300, 850),
            "years_in_business": rng.integers(1, 40, size=n),
        })
        logit = (
            -4.2
            + 3.2 * tmp["debt_ratio"]
            + 0.05 * tmp["days_late"]
            - 0.006 * (tmp["credit_score"] - 650)
            - 0.03 * tmp["years_in_business"]
        )
        proba = 1 / (1 + np.exp(-logit))
        tmp[TARGET] = rng.binomial(1, proba)
        tmp.to_csv(DATA_PATH, index=False)
df = pd.read_csv(DATA_PATH)
print(df.shape)
df.head()

## 1. Exploration
Comprends le déséquilibre des classes avant tout.

**Questions :** Quel est le taux de défaut ? Le problème est-il déséquilibré ? Y a-t-il des valeurs manquantes ?

In [ ]:
# Exploration de la cible et qualité des données
default_rate = df[TARGET].mean()
class_distribution = df[TARGET].value_counts(normalize=True).sort_index()
missing_values = df.isna().sum()

print(f"Taux de défaut : {default_rate:.2%}")
print("\nDistribution des classes :")
print(class_distribution)
print("\nValeurs manquantes par colonne :")
print(missing_values)


## 2. Split train / test stratifié
**Question :** pourquoi stratifier sur la cible quand les classes sont déséquilibrées ?


**Réponse :** on stratifie pour garder presque le même pourcentage de défauts dans train et test. Sans stratification, une classe minoritaire peut être sous-représentée dans le test, ce qui rend l'évaluation instable.


In [ ]:
from sklearn.model_selection import train_test_split

X, y = df[FEATURES], df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y
)

print(f"Train {len(X_train)} | Test {len(X_test)}")
print(f"Taux défaut train {y_train.mean():.2%} | test {y_test.mean():.2%}")


## 3. Préprocessing (ColumnTransformer)
One-hot sur les catégorielles, standardisation sur les numériques.

**Question :** pourquoi `handle_unknown="ignore"` pour le OneHotEncoder ?


**Réponse :** `handle_unknown="ignore"` évite une erreur si une nouvelle catégorie apparaît dans le test ou en production. Elle sera simplement encodée avec des zéros sur les colonnes one-hot connues.


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL_FEATURES),
        ("num", StandardScaler(), NUMERIC_FEATURES),
    ]
)

X_train_prepared = preprocessor.fit_transform(X_train)

print(f"Nombre de features initiales : {len(FEATURES)}")
print(f"Nombre de colonnes après preprocessing : {X_train_prepared.shape[1]}")


## 4. Baseline — LogisticRegression
Toujours une baseline simple AVANT les modèles complexes.

**Question :** que représente une probabilité prédite de 0.9 pour un client ?


**Réponse :** une probabilité prédite de `0.9` signifie que le modèle estime un risque de défaut très élevé pour ce client : environ 90% selon les patterns appris. Ce n'est pas une certitude, mais un score de risque.


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, roc_auc_score

log_reg_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("clf", LogisticRegression(
            max_iter=1000,
            class_weight="balanced",
            random_state=RANDOM_STATE
        )),
    ]
)

log_reg_model.fit(X_train, y_train)

y_pred_lr = log_reg_model.predict(X_test)
y_proba_lr = log_reg_model.predict_proba(X_test)[:, 1]

print(f"Logistic Regression - F1 : {f1_score(y_test, y_pred_lr):.4f}")
print(f"Logistic Regression - ROC-AUC : {roc_auc_score(y_test, y_proba_lr):.4f}")


## 5. RandomForest + XGBoost
Compare des modèles non linéaires à la baseline.

**Question :** pourquoi un arbre/forêt n'a pas besoin de standardisation, contrairement à la régression logistique ?


**Réponse :** les arbres comparent des seuils sur les variables, donc ils sont peu sensibles à l'échelle. La régression logistique optimise des coefficients numériques, donc les variables avec grandes valeurs peuvent dominer si elles ne sont pas standardisées.


In [ ]:
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb

scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

rf_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("clf", RandomForestClassifier(
            n_estimators=300,
            class_weight="balanced",
            random_state=RANDOM_STATE,
            n_jobs=-1
        )),
    ]
)

xgb_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("clf", xgb.XGBClassifier(
            n_estimators=300,
            max_depth=4,
            learning_rate=0.05,
            subsample=0.9,
            colsample_bytree=0.9,
            objective="binary:logistic",
            eval_metric="logloss",
            scale_pos_weight=scale_pos_weight,
            random_state=RANDOM_STATE,
            n_jobs=-1
        )),
    ]
)

models = {
    "Logistic Regression": log_reg_model,
    "Random Forest": rf_model,
    "XGBoost": xgb_model,
}

results = []
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]
    results.append({
        "model": name,
        "f1": f1_score(y_test, y_pred),
        "roc_auc": roc_auc_score(y_test, y_proba),
    })

results_df = pd.DataFrame(results).sort_values(by="f1", ascending=False)
results_df


## 6. Gérer le déséquilibre
**Question :** différence entre `class_weight='balanced'`, `scale_pos_weight` et SMOTE ? Lequel rééchantillonne réellement les données ?


**Réponse :** `class_weight='balanced'` change le poids des erreurs dans la fonction de coût. `scale_pos_weight` fait pareil pour XGBoost. SMOTE crée réellement de nouveaux exemples synthétiques de la classe minoritaire, donc c'est lui qui rééchantillonne les données.


In [ ]:
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.metrics import recall_score

smote_model = ImbPipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("smote", SMOTE(random_state=RANDOM_STATE)),
        ("clf", xgb.XGBClassifier(
            n_estimators=300,
            max_depth=4,
            learning_rate=0.05,
            subsample=0.9,
            colsample_bytree=0.9,
            objective="binary:logistic",
            eval_metric="logloss",
            random_state=RANDOM_STATE,
            n_jobs=-1
        )),
    ]
)

smote_model.fit(X_train, y_train)

y_pred_xgb = xgb_model.predict(X_test)
y_pred_smote = smote_model.predict(X_test)

print(f"Recall XGBoost sans SMOTE : {recall_score(y_test, y_pred_xgb):.4f}")
print(f"Recall XGBoost avec SMOTE : {recall_score(y_test, y_pred_smote):.4f}")


## 7. Métriques détaillées
**Question :** dans la détection de défaut/fraude, pourquoi le recall est-il souvent plus critique que l'accuracy ?


**Réponse :** dans la détection de défaut/fraude, un faux négatif signifie qu'on laisse passer un client risqué. Le recall mesure la capacité à détecter les vrais défauts, donc il est souvent plus important que l'accuracy sur un dataset déséquilibré.


In [ ]:
from sklearn.metrics import confusion_matrix, classification_report, precision_score, recall_score

# Choix du meilleur modèle selon le F1-score sur le test
best_model_name = results_df.iloc[0]["model"]
best_model = models[best_model_name]

# Si SMOTE donne un meilleur F1, on le choisit à la place
y_pred_smote = smote_model.predict(X_test)
y_proba_smote = smote_model.predict_proba(X_test)[:, 1]
smote_f1 = f1_score(y_test, y_pred_smote)

if smote_f1 > results_df.iloc[0]["f1"]:
    best_model_name = "XGBoost + SMOTE"
    best_model = smote_model
    y_pred_best = y_pred_smote
    y_proba_best = y_proba_smote
else:
    y_pred_best = best_model.predict(X_test)
    y_proba_best = best_model.predict_proba(X_test)[:, 1]

print(f"Meilleur modèle : {best_model_name}")
print("\nMatrice de confusion :")
print(confusion_matrix(y_test, y_pred_best))

print(f"\nPrecision : {precision_score(y_test, y_pred_best):.4f}")
print(f"Recall : {recall_score(y_test, y_pred_best):.4f}")
print(f"F1 : {f1_score(y_test, y_pred_best):.4f}")
print(f"ROC-AUC : {roc_auc_score(y_test, y_proba_best):.4f}")

print("\nClassification report :")
print(classification_report(y_test, y_pred_best))


## 8. Cross-validation stratifiée
**Question :** pourquoi une CV stratifiée donne une estimation plus robuste qu'un seul split ?


**Réponse :** une cross-validation stratifiée teste le modèle sur plusieurs découpages tout en gardant la distribution des classes. Elle donne une estimation plus robuste qu'un seul split train/test.


In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_score

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

cv_scores = cross_val_score(
    best_model,
    X,
    y,
    cv=cv,
    scoring="f1",
    n_jobs=-1
)

print(f"F1 CV : {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")
print(cv_scores)


## 9. Choix du seuil métier
Le seuil 0.5 n'est pas sacré. En assurance-crédit, manquer un défaut coûte cher.

**Question :** que se passe-t-il (precision vs recall) si on baisse le seuil de 0.5 à 0.3 ?


**Réponse :** baisser le seuil de 0.5 à 0.3 rend le modèle plus sensible : il détecte plus de défauts, donc le recall augmente. En contrepartie, il produit plus de faux positifs, donc la precision baisse généralement.


In [ ]:
from sklearn.metrics import precision_recall_curve
import matplotlib.pyplot as plt

y_proba = best_model.predict_proba(X_test)[:, 1]
thresholds = np.arange(0.1, 1.0, 0.1)

threshold_results = []
for threshold in thresholds:
    y_pred_threshold = (y_proba >= threshold).astype(int)
    threshold_results.append({
        "threshold": threshold,
        "precision": precision_score(y_test, y_pred_threshold, zero_division=0),
        "recall": recall_score(y_test, y_pred_threshold),
        "f1": f1_score(y_test, y_pred_threshold),
    })

threshold_df = pd.DataFrame(threshold_results)
display(threshold_df)

plt.figure(figsize=(8, 5))
plt.plot(threshold_df["threshold"], threshold_df["precision"], marker="o", label="Precision")
plt.plot(threshold_df["threshold"], threshold_df["recall"], marker="o", label="Recall")
plt.plot(threshold_df["threshold"], threshold_df["f1"], marker="o", label="F1")
plt.xlabel("Seuil")
plt.ylabel("Score")
plt.title("Impact du seuil sur precision, recall et F1")
plt.legend()
plt.grid(True)
plt.show()

# Exemple de choix métier : maximiser le recall avec precision >= 30%
acceptable_precision = 0.30
valid_thresholds = threshold_df[threshold_df["precision"] >= acceptable_precision]

if len(valid_thresholds) > 0:
    chosen = valid_thresholds.sort_values(by="recall", ascending=False).iloc[0]
    print(
        f"Seuil choisi : {chosen['threshold']:.1f} | "
        f"Precision : {chosen['precision']:.2%} | "
        f"Recall : {chosen['recall']:.2%} | "
        f"F1 : {chosen['f1']:.2%}"
    )
else:
    print("Aucun seuil ne respecte la contrainte de precision.")


---
**Auto-évaluation :** une fois terminé, compare ton raisonnement avec [`solutions/classification_xgboost.py`](solutions/classification_xgboost.py) (`python solutions/classification_xgboost.py` depuis `machine_learning/`).